# Lab 3: Data Cleaning & Preprocessing

Welcome to Lab 3! Real-world data is almost never ready to use straight away — it usually has missing values, duplicate rows, inconsistent formatting, wrong data types, and outliers. This lab walks through identifying and fixing each of these problems using Pandas and NumPy.

**After this lab you will be able to:**
- Identify common data quality issues in a real dataset
- Handle missing values using multiple strategies
- Remove duplicate records
- Standardize inconsistent text and date formats
- Fix incorrect data types
- Detect and handle outliers

**Instructions:**
- Write your code between the `### YOUR CODE HERE ###` and `### END ###` markers.
- Run each cell with **Shift + Enter**.
- Compare your output with the **Expected Output** shown below each exercise where provided.

## Why Data Cleaning Matters

Before any model can learn from data, the data itself needs to be trustworthy. Messy data leads directly to wrong conclusions and poor model performance — a model cannot tell the difference between a genuine value and a data-entry mistake unless you clean it first. In most real projects, data cleaning and preprocessing take up more time than model building itself.

The most common problems you will run into are:

- **Missing values** — some cells are simply empty (`NaN`), often because of incomplete data entry.
- **Duplicate records** — the same row appears more than once, which can bias any statistics or model trained on it.
- **Inconsistent formatting** — the same category written in different ways (`"IT"`, `"it"`, `"IT "`), which a computer treats as entirely different values.
- **Incorrect data types** — numbers stored as text (e.g., `"$45,000"` instead of `45000`), which blocks any numeric calculation.
- **Outliers** — values far outside the expected range (e.g., an age of 200), which can distort averages and model training.

We will fix each of these, one at a time, using the dataset below.

## Loading the Dataset

For this lab, you have been given a file named **employee_records_messy.csv**. It is a small, intentionally messy HR dataset containing exactly the kinds of issues described above.

**Upload the file to this Colab session** using the cell below. When you run it, a button will appear — click it and select `employee_records_messy.csv` from your computer.

Note: Files uploaded this way only last for the current session. If your Colab runtime disconnects, you will need to re-run this cell and upload the file again.

In [2]:
from google.colab import files

uploaded = files.upload()   # click the button that appears and select employee_records_messy.csv

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("employee_records_messy.csv")
df.head(10)

## Step 1 — Inspecting the Data

Before fixing anything, always inspect the dataset first: how many rows and columns it has, what data types each column has, and how many missing values exist.

In [ ]:
df.shape

In [ ]:
df.info()

**Exercise:** Print the number of missing values in each column, and separately, the number of fully duplicated rows in the dataset.

In [ ]:
missing_counts = df.isnull().sum()
duplicate_count = df.duplicated().sum()

print(missing_counts)
print("Duplicate rows:", duplicate_count)

## Step 2 — Removing Duplicates

Exact duplicate rows add no new information and can skew any summary statistics or model trained on the data.

**Exercise:** Create `df_no_dupes`, a copy of `df` with all fully duplicated rows removed. Keep the first occurrence of each.

In [ ]:
df_no_dupes = df.drop_duplicates(keep='first')

print("Before:", df.shape[0], "rows | After:", df_no_dupes.shape[0], "rows")

## Step 3 — Handling Missing Values

There is no single correct way to handle missing values — the right strategy depends on the column and the situation:

- **Drop rows** when very few rows have missing values and losing them won't hurt the analysis.
- **Fill with the mean/median** for numeric columns, when a reasonable estimate is better than losing the row entirely.
- **Fill with a placeholder** (e.g., `"Unknown"`) for categorical/text columns, when the missing information is itself meaningful.

**Exercise:** On `df_no_dupes`, fill missing `age` values with the column's median, fill missing `performance_score` values with the column's mean, and drop any remaining rows where `email` is still missing. Store the result in `df_clean`.

In [ ]:
df_clean = df_no_dupes.copy()

df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
df_clean['performance_score'] = df_clean['performance_score'].fillna(df_clean['performance_score'].mean())
df_clean = df_clean.dropna(subset=['email'])

print(df_clean.isnull().sum())

## Step 4 — Standardizing Inconsistent Text

Look closely at the `name` and `department` columns — the same value often appears with different capitalization or extra whitespace (e.g., `"bilal ahmed"` vs `"BILAL AHMED"`, `"IT "` with a trailing space). To a computer, these are different strings unless you standardize them.

**Exercise:** On `df_clean`, strip extra whitespace and convert `name` to title case, and strip whitespace and convert `department` to title case as well.

In [ ]:
df_clean['name'] = df_clean['name'].str.strip().str.title()
df_clean['department'] = df_clean['department'].str.strip().str.title()

print(df_clean["department"].unique())

**Expected Output (department values, order may vary):**
```
['Sales' 'It' 'Hr' 'Marketing' 'Finance']
```

## Step 5 — Fixing Incorrect Data Types

The `salary` column is a mix of plain numbers and text values like `"$61,000"`. Before any numeric calculation can be done, every value needs to be a proper number.

**Exercise:** Write a function `clean_salary(value)` that removes `$` and `,` characters (if present) and converts the result to a float, and use `.apply()` to create a cleaned `salary` column in `df_clean`.

In [ ]:
def clean_salary(value):
    if pd.isna(value):
        return value
    return float(str(value).replace('$', '').replace(',', ''))

df_clean["salary"] = df_clean["salary"].apply(clean_salary)
print(df_clean["salary"].dtype)
print(df_clean["salary"].head(10))

## Standardizing Dates

The `join_date` column has several different date formats (`"2021-03-15"`, `"15/06/2020"`, `"March 3, 2022"`). Pandas can parse most common formats automatically with `pd.to_datetime()` using `errors="coerce"`, which turns anything it cannot parse into `NaT` (a missing date) instead of crashing.

In [ ]:
df_clean["join_date"] = pd.to_datetime(df_clean["join_date"], errors="coerce")
df_clean["join_date"].head(10)

## Step 6 — Detecting Outliers

An outlier is a value far outside the range you would reasonably expect — for example, an employee age of 200. One common way to detect outliers is the IQR (Interquartile Range) method: values below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR` are flagged.

**Exercise:** Using the IQR method on the `age` column, identify which rows are outliers and store them in `age_outliers`. Then replace outlier ages with the column's median.

In [ ]:
Q1 = df_clean["age"].quantile(0.25)
Q3 = df_clean["age"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

age_outliers = df_clean[(df_clean["age"] < lower_bound) | (df_clean["age"] > upper_bound)]

print(age_outliers[["employee_id", "age"]])

In [ ]:
median_age = df_clean["age"].median()
df_clean.loc[(df_clean["age"] < lower_bound) | (df_clean["age"] > upper_bound), "age"] = median_age
df_clean["age"].describe()

**What to remember from this lab:**
- Always inspect a dataset (`.info()`, `.isnull().sum()`, `.duplicated().sum()`) before doing anything else
- There is no single "correct" way to handle missing values — the right strategy depends on context
- Inconsistent text formatting must be standardized before grouping or filtering by it
- Data types must be correct before any numeric operation will work
- The IQR method is one simple, common way to flag outliers

## Lab Tasks

Complete the following tasks in this notebook, below this cell, using `df_clean` (or the original `df` where specified).

1. **Full Pipeline Recap:** In a markdown cell, write 3-4 sentences summarizing, in your own words, the full cleaning pipeline you just performed on this dataset (duplicates → missing values → text → types → dates → outliers).
2. **Salary Outliers:** Repeat the IQR outlier-detection method from Step 6, but this time apply it to the cleaned `salary` column. Print any rows flagged as outliers.
3. **Department Summary:** Using `df_clean`, compute the average `salary` and average `performance_score` per `department`, in a single `.groupby().agg()` call.
4. **Export the Cleaned Data:** Save `df_clean` to a new file named `employee_records_clean.csv` using `.to_csv()`, without the index column.
5. **Reflection:** In a markdown cell, note which cleaning step you found most challenging and why.

In [ ]:
# Task 1
# The data cleaning pipeline started with identifying missing values and duplicate rows.
# Then, exact duplicates were removed to prevent bias. Missing values in numerical columns like 'age' and 'performance_score' were imputed using the median and mean, respectively, while rows with missing 'email' were dropped.
# Text data in 'name' and 'department' was standardized by stripping whitespace and converting to title case.
# Finally, the 'salary' column was cleaned of currency symbols and converted to numerical type, and the 'join_date' was converted to datetime format before identifying and capping outliers in 'age'.

In [ ]:
# Task 2
Q1_sal = df_clean["salary"].quantile(0.25)
Q3_sal = df_clean["salary"].quantile(0.75)
IQR_sal = Q3_sal - Q1_sal
lower_bound_sal = Q1_sal - 1.5 * IQR_sal
upper_bound_sal = Q3_sal + 1.5 * IQR_sal

salary_outliers = df_clean[(df_clean["salary"] < lower_bound_sal) | (df_clean["salary"] > upper_bound_sal)]
print("Salary Outliers:")
print(salary_outliers)

In [ ]:
# Task 3
department_summary = df_clean.groupby('department').agg(
    avg_salary=('salary', 'mean'),
    avg_performance=('performance_score', 'mean')
)
print(department_summary)

In [ ]:
# Task 4
df_clean.to_csv('employee_records_clean.csv', index=False)

*Task 5 Reflection:*

The most challenging step was deciding how to handle missing values, particularly for the 'age' and 'performance_score' columns. Different imputation techniques (like mean vs. median) can significantly affect the data distribution. Dropping rows, on the other hand, risks losing valuable information, which is why it was only applied to the 'email' column where imputation wasn't possible. Standardizing the salary strings into a numeric type also required careful attention to ensure all special characters were correctly removed.